# Brain Tumor MRI — RAG Clinical Decision Support Setup

This notebook builds a **Retrieval-Augmented Generation (RAG)** pipeline that provides
evidence-based clinical guidelines when the model classifies a brain MRI scan.

**Stack:**
- `sentence-transformers` — embed medical guideline text
- `chromadb` — persistent vector store for semantic retrieval
- `all-MiniLM-L6-v2` — compact, fast embedding model (384-dim)

**Workflow:**
1. Install dependencies
2. Mount Google Drive, set paths
3. Load and parse `medical_guidelines.txt` (split on `---` separator)
4. Embed each section with SentenceTransformer
5. Store in ChromaDB PersistentClient
6. Test retrieval queries for each tumor type
7. Full clinical decision support demo
8. Verify persistence

---

## Step 1: Install Dependencies

In [ ]:
!pip install chromadb sentence-transformers --quiet

import chromadb
import sentence_transformers

print(f'chromadb version          : {chromadb.__version__}')
print(f'sentence-transformers ver : {sentence_transformers.__version__}')
print('Dependencies ready.')

## Step 2: Mount Google Drive and Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

GUIDELINES_PATH = '/content/drive/MyDrive/BrainTumorModel/medical_guidelines.txt'
CHROMA_DB_PATH  = '/content/drive/MyDrive/BrainTumorModel/chroma_db'
COLLECTION_NAME = 'brain_tumor_guidelines'

os.makedirs(CHROMA_DB_PATH, exist_ok=True)

print(f'Guidelines path : {GUIDELINES_PATH}')
print(f'ChromaDB path   : {CHROMA_DB_PATH}')
print(f'Guidelines exist: {os.path.exists(GUIDELINES_PATH)}')

## Step 3: Create Sample medical_guidelines.txt (if not already present)

Each section is separated by `---` on its own line.  
If you already have a curated guidelines file in Drive, skip this cell.

In [ ]:
SAMPLE_GUIDELINES = """GLIOMA CLINICAL GUIDELINES
Gliomas are primary brain tumors arising from glial cells, classified by the WHO grading system (Grade I-IV). High-grade gliomas (Grade III-IV), including glioblastoma multiforme (GBM), are aggressive and require urgent neurosurgical evaluation. MRI characteristics include irregular enhancement, mass effect, peritumoral edema, and infiltrative margins. Standard of care for GBM includes maximal safe surgical resection followed by concurrent temozolomide chemotherapy and radiotherapy (Stupp protocol). Bevacizumab may be used for recurrent disease. Median survival for GBM is 14-16 months with treatment. Molecular markers IDH1/IDH2 mutation and MGMT promoter methylation are critical prognostic and predictive factors.
Recommended actions: Urgent neurosurgery referral, MRI with and without contrast, stereotactic biopsy if resection not feasible, molecular pathology panel (IDH, MGMT, 1p/19q co-deletion), multidisciplinary tumor board review.
---
MENINGIOMA CLINICAL GUIDELINES
Meningiomas arise from the meninges (arachnoid cap cells) and account for approximately 37% of primary brain tumors. The majority (80-85%) are benign (WHO Grade I). MRI hallmarks include a dural-based extra-axial mass with homogeneous enhancement, a dural tail sign, and possible calcification. Small asymptomatic meningiomas in elderly patients may be managed with watchful waiting and serial MRI at 6-12 month intervals. Surgical resection is the primary treatment for symptomatic tumors; extent of resection is graded by the Simpson scale. Stereotactic radiosurgery (SRS) is effective for tumors under 3 cm or surgical residuals. Grade II and III meningiomas carry higher recurrence risk and may require adjuvant radiotherapy.
Recommended actions: Serial MRI surveillance for small asymptomatic tumors, neurosurgery consultation for symptomatic or growing lesions, SRS for tumors under 3cm, histopathology and WHO grading post-resection.
---
PITUITARY TUMOR CLINICAL GUIDELINES
Pituitary adenomas represent the most common sellar lesion and account for 10-15% of intracranial neoplasms. They are classified as microadenomas (<10 mm) or macroadenomas (>=10 mm). Functional adenomas secrete hormones: prolactinomas are the most common (treated medically with dopamine agonists such as cabergoline), followed by GH-secreting (acromegaly) and ACTH-secreting (Cushing disease) adenomas. Non-functioning adenomas are managed by observation or surgery. MRI reveals a sellar mass with variable enhancement; extension into the cavernous sinus or optic chiasm compression requires urgent surgical decompression via transsphenoidal approach. Visual field testing is essential when suprasellar extension is present. Full pituitary hormone assessment is mandatory.
Recommended actions: Endocrinology referral, full pituitary hormone panel (cortisol, TSH, IGF-1, prolactin, FSH/LH), visual field examination, dedicated sella MRI, neuro-ophthalmology if chiasm involvement, neurosurgery for macroadenomas with visual compromise.
---
NO TUMOR — NORMAL BRAIN MRI GUIDELINES
A normal brain MRI in the context of neurological symptoms requires clinical correlation. Common differential diagnoses when no structural lesion is found include functional neurological disorder, migraine, early demyelinating disease, cerebrovascular risk factors, metabolic encephalopathy, and psychiatric conditions. Recommended workup depends on presenting symptoms: new-onset headache warrants migraine evaluation and vascular imaging; cognitive symptoms require neuropsychological assessment; focal neurological deficits warrant EEG and CSF analysis. Follow-up MRI with contrast may be indicated if symptoms are progressive or atypical. Reassurance and conservative management are appropriate for incidental findings of no clinical significance (e.g., minor white matter T2 changes in older patients).
Recommended actions: Clinical correlation with presenting symptoms, neurology consultation, CSF analysis if demyelinating disease suspected, vascular imaging for headache or TIA symptoms, EEG for episodic neurological events, repeat MRI in 3-6 months if symptoms persist.
"""

if not os.path.exists(GUIDELINES_PATH):
    with open(GUIDELINES_PATH, 'w', encoding='utf-8') as f:
        f.write(SAMPLE_GUIDELINES)
    print(f'Sample guidelines created at: {GUIDELINES_PATH}')
else:
    print(f'Using existing guidelines file: {GUIDELINES_PATH}')

## Step 4: Load and Parse medical_guidelines.txt

Split the document into sections using `---` as the delimiter. Each section becomes one document in the vector store.

In [ ]:
with open(GUIDELINES_PATH, 'r', encoding='utf-8') as f:
    raw_text = f.read()

# Split on --- separator and strip whitespace
sections = [s.strip() for s in raw_text.split('---') if s.strip()]

print(f'Total sections parsed: {len(sections)}')
print()
for i, section in enumerate(sections):
    first_line = section.split('\n')[0]
    char_count = len(section)
    print(f'  Section {i+1}: {first_line}  [{char_count} chars]')

print()
print('=== Preview: Section 1 (first 300 chars) ===')
print(sections[0][:300] + '...')

## Step 5: Load SentenceTransformer — all-MiniLM-L6-v2

`all-MiniLM-L6-v2` produces 384-dimensional embeddings optimized for semantic similarity tasks.
It is fast, lightweight, and well-suited for retrieval from small-to-medium document collections.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBEDDING_MODEL = 'all-MiniLM-L6-v2'
print(f'Loading SentenceTransformer: {EMBEDDING_MODEL} ...')
embedder = SentenceTransformer(EMBEDDING_MODEL)

# Sanity check
test_embedding = embedder.encode('brain tumor MRI classification')
print(f'Model loaded successfully.')
print(f'Embedding dimension : {len(test_embedding)}')
print(f'Embedding dtype     : {test_embedding.dtype}')

## Step 6: Embed All Sections

In [ ]:
import time

print('Embedding guideline sections...')
t0 = time.time()
embeddings = embedder.encode(sections, show_progress_bar=True, batch_size=8)
elapsed = time.time() - t0

print(f'\nEmbedding complete in {elapsed:.2f}s')
print(f'Embeddings shape : {embeddings.shape}')  # (num_sections, 384)
print(f'Sample norm      : {np.linalg.norm(embeddings[0]):.4f}')

## Step 7: Store Embeddings in ChromaDB PersistentClient

Each section is stored with:
- A unique string `id`
- The raw text as `document`
- The 384-dim vector as `embedding`
- Metadata: `tumor_class` and `section_idx`

Cosine similarity is used so semantic closeness is measured regardless of vector magnitude.

In [ ]:
import chromadb

# Create / connect to persistent client
client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

# Drop existing collection to allow re-running cleanly
try:
    client.delete_collection(COLLECTION_NAME)
    print(f'Deleted existing collection: {COLLECTION_NAME}')
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={'hnsw:space': 'cosine'}
)

# Map class names from section titles
CLASS_KEYWORDS = {
    'glioma':     'glioma',
    'meningioma': 'meningioma',
    'pituitary':  'pituitary',
    'no_tumor':   'no tumor',
}

doc_ids        = []
doc_texts      = []
doc_embeddings = []
doc_metadatas  = []

for i, (section, embedding) in enumerate(zip(sections, embeddings)):
    first_line   = section.split('\n')[0].lower()
    tumor_class  = 'unknown'
    for cls, keyword in CLASS_KEYWORDS.items():
        if keyword in first_line:
            tumor_class = cls
            break

    doc_ids.append(f'section_{i}')
    doc_texts.append(section)
    doc_embeddings.append(embedding.tolist())
    doc_metadatas.append({'tumor_class': tumor_class, 'section_idx': i})

collection.add(
    ids=doc_ids,
    documents=doc_texts,
    embeddings=doc_embeddings,
    metadatas=doc_metadatas
)

print(f'\nCollection "{COLLECTION_NAME}" created.')
print(f'Documents stored : {collection.count()}')
print(f'Persisted at     : {CHROMA_DB_PATH}')
print()
for doc_id, meta in zip(doc_ids, doc_metadatas):
    print(f'  {doc_id}  →  tumor_class: {meta["tumor_class"]}')

## Step 8: Define Retrieval Function

In [ ]:
def retrieve_guidelines(query: str, n_results: int = 2) -> list:
    """
    Embed the query and retrieve the top-n most similar guideline sections.

    Args:
        query     : Clinical question or tumor class name.
        n_results : Number of guideline sections to return.

    Returns:
        List of dicts with keys: id, document, metadata, distance.
    """
    query_embedding = embedder.encode(query).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=['documents', 'metadatas', 'distances']
    )
    output = []
    for i in range(len(results['ids'][0])):
        output.append({
            'id':       results['ids'][0][i],
            'document': results['documents'][0][i],
            'metadata': results['metadatas'][0][i],
            'distance': round(results['distances'][0][i], 4),
        })
    return output

print('retrieve_guidelines() defined.')

## Step 9: Test Retrieval Queries for Each Tumor Type

Verify that clinical queries for each class retrieve the correct guideline section.

In [ ]:
test_queries = [
    ('glioma',     'glioblastoma treatment chemotherapy radiation Stupp protocol IDH'),
    ('meningioma', 'meningioma dural-based tumor surgery radiosurgery Simpson scale'),
    ('pituitary',  'pituitary adenoma prolactin hormone transsphenoidal resection'),
    ('no_tumor',   'normal brain MRI no structural lesion clinical follow-up'),
]

print('Retrieval validation:')
print('=' * 65)
all_passed = True
for expected_class, query in test_queries:
    results         = retrieve_guidelines(query, n_results=1)
    retrieved_class = results[0]['metadata']['tumor_class']
    dist            = results[0]['distance']
    passed          = (retrieved_class == expected_class)
    status          = 'PASS' if passed else 'FAIL'
    if not passed:
        all_passed = False
    title = results[0]['document'].split('\n')[0]
    print(f'[{status}] Expected: {expected_class:<12} Got: {retrieved_class:<12} dist: {dist}')
    print(f'      Title: {title}')
    print()

if all_passed:
    print('All 4 retrieval tests PASSED — RAG correctly maps each query to the right tumor class.')
else:
    print('Some tests failed. Review the guidelines text and query strings.')

## Step 10: Clinical Decision Support — Full Pipeline Demo

This function simulates the end-to-end integration: given a ResNet18 prediction, it retrieves and presents the relevant clinical guidelines.

In [ ]:
def clinical_decision_support(
    predicted_class: str,
    confidence: float,
    n_guidelines: int = 1
) -> None:
    """
    Given a model prediction, retrieve and display relevant clinical guidelines.

    Args:
        predicted_class : One of: glioma, meningioma, pituitary, no_tumor.
        confidence      : Softmax confidence from ResNet18 (0.0 to 1.0).
        n_guidelines    : Number of guideline sections to display.
    """
    QUERY_MAP = {
        'glioma':     'glioma glioblastoma treatment surgery chemotherapy IDH MGMT',
        'meningioma': 'meningioma brain tumor management surgery stereotactic radiosurgery',
        'pituitary':  'pituitary adenoma endocrine hormone transsphenoidal evaluation',
        'no_tumor':   'normal brain MRI no structural lesion clinical correlation workup',
    }

    query   = QUERY_MAP.get(predicted_class, predicted_class)
    results = retrieve_guidelines(query, n_results=n_guidelines)

    conf_level = ('HIGH' if confidence >= 0.85
                  else 'MODERATE' if confidence >= 0.60
                  else 'LOW')

    print('=' * 70)
    print('   CLINICAL DECISION SUPPORT REPORT')
    print('=' * 70)
    print(f'  AI Diagnosis     : {predicted_class.upper()}')
    print(f'  Confidence       : {confidence*100:.1f}%  [{conf_level}]')

    if conf_level == 'LOW':
        print()
        print('  *** WARNING: Low-confidence prediction. ***')
        print('  Expert radiologist review is strongly recommended.')

    print()
    print('  RETRIEVED CLINICAL GUIDELINES:')
    print('  ' + '-' * 66)

    for i, result in enumerate(results, 1):
        print(f'\n  [Guideline {i}]  Section: {result["id"]}'
              f'  |  Class: {result["metadata"]["tumor_class"]}'
              f'  |  Similarity distance: {result["distance"]}\n')
        for line in result['document'].split('\n'):
            if line.strip():
                print(f'    {line}')

    print()
    print('  DISCLAIMER: This is an AI-assisted tool. All outputs must be')
    print('  reviewed by a qualified clinician before any clinical action.')
    print('=' * 70)


# Demo 1: high-confidence glioma
clinical_decision_support('glioma', confidence=0.93)

In [ ]:
# Demo 2: moderate-confidence meningioma
clinical_decision_support('meningioma', confidence=0.68)

In [ ]:
# Demo 3: high-confidence pituitary tumor
clinical_decision_support('pituitary', confidence=0.89)

In [ ]:
# Demo 4: no tumor detected
clinical_decision_support('no_tumor', confidence=0.96)

## Step 11: Free-Form Clinical Query Test

Verify that the system returns clinically sensible results for natural language questions.

In [ ]:
free_form_queries = [
    ('glioma',     'What is the standard chemotherapy protocol after brain tumor resection?'),
    ('pituitary',  'Patient has elevated prolactin levels and bitemporal visual field defects'),
    ('meningioma', 'Dural-based homogeneous enhancing mass with dural tail sign on MRI'),
    ('no_tumor',   'Patient has persistent headaches but no lesion visible on brain scan'),
    ('glioma',     'IDH mutation and MGMT methylation status in brain tumors'),
]

print('Free-form clinical query retrieval validation:')
print('=' * 70)
for expected_class, query in free_form_queries:
    results         = retrieve_guidelines(query, n_results=1)
    retrieved_class = results[0]['metadata']['tumor_class']
    dist            = results[0]['distance']
    match           = 'PASS' if retrieved_class == expected_class else 'FAIL'
    print(f'[{match}] Q: "{query[:55]}..."')
    print(f'       Expected: {expected_class}  |  Got: {retrieved_class}  |  dist: {dist}')
    print()

## Step 12: Verify ChromaDB Persistence

Confirm that the vector store reloads correctly from disk without re-embedding.

In [ ]:
import chromadb

# Reload client from disk (simulate a new session)
client2     = chromadb.PersistentClient(path=CHROMA_DB_PATH)
collection2 = client2.get_collection(COLLECTION_NAME)

print(f'Reloaded collection : "{COLLECTION_NAME}"')
print(f'Documents in store  : {collection2.count()}')

# Run a test query directly through the reloaded collection
test_emb = embedder.encode('glioma treatment temozolomide').tolist()
result   = collection2.query(
    query_embeddings=[test_emb],
    n_results=1,
    include=['metadatas', 'distances']
)
retrieved_cls = result['metadatas'][0][0]['tumor_class']
dist_val      = result['distances'][0][0]

print(f'\nTest query: "glioma treatment temozolomide"')
print(f'  Retrieved class : {retrieved_cls}')
print(f'  Distance        : {dist_val:.4f}')

assert retrieved_cls == 'glioma', f'Expected glioma, got {retrieved_cls}'
assert collection2.count() == len(sections), 'Document count mismatch!'

print('\nChromaDB persistence verified successfully.')

## Summary

The RAG clinical decision support pipeline is fully operational.

| Component | Details |
|-----------|----------|
| Embedding model | `all-MiniLM-L6-v2` (384-dim, cosine similarity) |
| Vector store | ChromaDB PersistentClient (Google Drive) |
| Document source | `medical_guidelines.txt` |
| Splitting strategy | `---` separator |
| Collection name | `brain_tumor_guidelines` |
| Sections indexed | 4 (glioma, meningioma, pituitary, no_tumor) |

**Integration with the classifier:**
After ResNet18 produces a prediction + confidence score (notebook 02/03), pass both to `clinical_decision_support(predicted_class, confidence)` to surface the corresponding evidence-based clinical guidance.

**Important:** This system is an AI-assisted decision-support tool. All outputs must be reviewed and validated by qualified medical professionals before any clinical action is taken.